In [1]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## 데이터 로드 및 전처리

In [7]:
df_covid19 = pd.read_csv("owid-covid-data.csv", parse_dates=['date'])

In [8]:
start_date = pd.Timestamp("2021-10-03")
end_date = start_date + pd.Timedelta(days=700) # 웹상 데이터가 변경되어, 기존 일별 확진자수가 주별 확진자수로 함쳐진 관계로, 구현의 편의상 100일이 아닌 100주 기간으로 진행

df_covid19_100 = (
    df_covid19[
        (df_covid19["iso_code"].isin(['KOR', 'OWID_ASI', 'OWID_EUR', 'OWID_OCE', 'OWID_NAM', 'OWID_SAM', 'OWID_AFR'])) &
        (df_covid19["date"] >= start_date) & (df_covid19["date"] <= end_date) &
        (df_covid19["new_cases"] != 0)
    ]
    .dropna(subset = ["new_cases"])
    .copy()
)


location_map = {
    'South Korea': '한국',
    'Asia': '아시아',
    'Europe': '유럽',
    'Oceania': '오세아니아',
    'North America': '북미',
    'South America': '남미',
    'Africa': '아프리카'
}
df_covid19_100["location"] = df_covid19_100["location"].replace(location_map)

ordered_locations = ['한국', '아시아', '유럽', '북미', '남미', '아프리카', '오세아니아']
df_covid19_100["location"] = pd.Categorical(df_covid19_100["location"], categories=ordered_locations, ordered=True)

df_covid19_100 = df_covid19_100.sort_values(by="date")
df_covid19_100

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
2311,OWID_AFR,NaN,아프리카,2021-10-03,8425012.0,69112.0,9873.14,212408.0,2412.0,344.57,...,NaN,NaN,NaN,NaN,NaN,1426736614,NaN,NaN,NaN,NaN
277822,OWID_NAM,NaN,북미,2021-10-03,52589480.0,923259.0,131894.14,1081581.0,17840.0,2548.57,...,NaN,NaN,NaN,NaN,NaN,600323657,NaN,NaN,NaN,NaN
358201,OWID_SAM,NaN,남미,2021-10-03,37911866.0,169100.0,24157.14,1157222.0,4597.0,656.71,...,NaN,NaN,NaN,NaN,NaN,436816679,NaN,NaN,NaN,NaN
20729,OWID_ASI,NaN,아시아,2021-10-03,76081520.0,1019817.0,145688.14,1129136.0,13947.0,1992.43,...,NaN,NaN,NaN,NaN,NaN,4721383370,NaN,NaN,NaN,NaN
359875,KOR,Asia,한국,2021-10-03,319777.0,16224.0,2317.71,2513.0,57.0,8.14,...,40.9,NaN,12.27,83.03,0.92,51815808,126.00,0.02,6.04,2.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278522,OWID_NAM,NaN,북미,2023-09-03,124342055.0,2494.0,356.29,1613444.0,1243.0,177.57,...,NaN,NaN,NaN,NaN,NaN,600323657,NaN,NaN,NaN,NaN
3011,OWID_AFR,NaN,아프리카,2023-09-03,13109512.0,596.0,85.14,259017.0,0.0,0.00,...,NaN,NaN,NaN,NaN,NaN,1426736614,NaN,NaN,NaN,NaN
358901,OWID_SAM,NaN,남미,2023-09-03,68571818.0,902.0,128.86,1352318.0,7.0,1.00,...,NaN,NaN,NaN,NaN,NaN,436816679,NaN,NaN,NaN,NaN
119906,OWID_EUR,NaN,유럽,2023-09-03,249247766.0,56967.0,8138.14,2077473.0,503.0,71.86,...,NaN,NaN,NaN,NaN,NaN,744807803,NaN,NaN,NaN,NaN


In [64]:
df_covid19_100_wide = df_covid19_100[['date', 'location', 'new_cases', 'people_fully_vaccinated_per_hundred']].copy()
df_covid19_100_wide.rename(columns={'new_cases': '확진자',
                                    'people_fully_vaccinated_per_hundred': '백신접종완료자'}, inplace=True)

df_covid19_100_wide = df_covid19_100_wide.pivot(index='date', columns='location', values=['확진자', '백신접종완료자'])
df_covid19_100_wide.columns = [f"{col[0]}_{col[1]}" for col in df_covid19_100_wide.columns]

df_covid19_100_wide = df_covid19_100_wide.sort_values(by="date").reset_index()

df_covid19_100_wide.head()

,date,확진자_한국,확진자_아시아,확진자_유럽,확진자_북미,확진자_남미,확진자_아프리카,확진자_오세아니아,백신접종완료자_한국,백신접종완료자_아시아,백신접종완료자_유럽,백신접종완료자_북미,백신접종완료자_남미,백신접종완료자_아프리카,백신접종완료자_오세아니아
0,2021-10-03,16224.0,1019817.0,909344.0,923259.0,169100.0,69112.0,20191.0,51.97,37.06,51.96,48.33,41.82,4.30,32.51
1,2021-10-10,13039.0,905692.0,1005831.0,819132.0,153169.0,53609.0,19859.0,58.56,38.36,52.73,49.34,44.15,4.56,36.25
2,2021-10-17,10629.0,843548.0,1159084.0,691289.0,124720.0,46062.0,18904.0,63.84,39.25,53.26,50.25,46.05,4.88,40.29
3,2021-10-24,9644.0,776191.0,1437425.0,597748.0,137108.0,39367.0,18688.0,69.23,41.02,53.69,51.14,47.90,5.23,43.57
4,2021-10-31,13296.0,745385.0,1590405.0,603822.0,136522.0,37830.0,13886.0,74.40,42.35,54.36,52.72,49.99,5.66,46.31


# 1. Plotly

In [12]:
#!pip3 install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 5.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [13]:
import plotly.graph_objects as go

## Plotly (빈) 객체생성
fig = go.Figure()
fig.show()

In [65]:
# Figure 객체에 Trace 속성 입력
# 첫번째 방법
go.Figure(
    data = [{
        'type' : 'scatter', # 산점도
        'x' : df_covid19_100.loc[df_covid19_100['location'] == '한국','date'],      # x축
        'y' : df_covid19_100.loc[df_covid19_100['location'] == '한국','new_cases'], # y축
        'line' : {'color' : '#5E88FC', 'dash' : 'dash'}}
    ]).show()

In [70]:
# Layered Structure의 특성을 활용하여 add_trace() Method를 활용
# 두번째 방법
fig = go.Figure()
fig.add_trace({
    'type' : 'scatter', # 산점도
    'x' : df_covid19_100.loc[df_covid19_100['location'] == '한국','date'],      # x축
    'y' : df_covid19_100.loc[df_covid19_100['location'] == '한국','new_cases'], # y축
    'line' : {'color' : '#5E88FC', 'dash' : 'dash'} # line 색상 등
})
fig.show()

In [16]:
fig.add_trace({ # Trace2 -  points
        'type' : 'scatter', # 관계형
        'mode' : 'markers', # 산점도
        'x' : df_covid19_100.loc[df_covid19_100['location'] == '한국','date'],      # x축
        'y' : df_covid19_100.loc[df_covid19_100['location'] == '한국','new_cases'], # y축
        'marker' : {'color' : '#264E86'}
})

fig.show()

In [17]:
fig.update_layout({ # layout
        'title' : "코로나19 발생 현황",
        'xaxis' : {'title' : "날짜", 'showgrid' : False},
        'yaxis' : {'title' : "확진자수"},
        'margin' : {'t' : 50, 'b' : 25, 'l' : 25, 'r' : 25}, # 여백
        'showlegend' : False
        })

fig.show()

In [18]:
# plotly 객체의 구조 확인
print(fig)

Figure({
    'data': [{'line': {'color': '#5E88FC', 'dash': 'dash'},
              'type': 'scatter',
              'x': array(['2021-10-03T00:00:00.000000', '2021-10-10T00:00:00.000000',
                          '2021-10-17T00:00:00.000000', '2021-10-24T00:00:00.000000',
                          '2021-10-31T00:00:00.000000', '2021-11-07T00:00:00.000000',
                          '2021-11-14T00:00:00.000000', '2021-11-21T00:00:00.000000',
                          '2021-11-28T00:00:00.000000', '2021-12-05T00:00:00.000000',
                          '2021-12-12T00:00:00.000000', '2021-12-19T00:00:00.000000',
                          '2021-12-26T00:00:00.000000', '2022-01-02T00:00:00.000000',
                          '2022-01-09T00:00:00.000000', '2022-01-16T00:00:00.000000',
                          '2022-01-23T00:00:00.000000', '2022-01-30T00:00:00.000000',
                          '2022-02-06T00:00:00.000000', '2022-02-13T00:00:00.000000',
                          '2022-02-20T

In [19]:
# Dictionary 생성
{
 'title' : "코로나19 발생 현황",
 'xaxis' : {'title' : "날짜", 'showgrid' : False},
 'yaxis' : {'title' : "확진자수"},
 'margin' : {'t' : 50, 'b' : 25, 'l' : 25, 'r' : 25},
 'showlegend' : False
}

{'title': '코로나19 발생 현황',
 'xaxis': {'title': '날짜', 'showgrid': False},
 'yaxis': {'title': '확진자수'},
 'margin': {'t': 50, 'b': 25, 'l': 25, 'r': 25},
 'showlegend': False}

In [20]:
dict(title = "코로나19 발생 현황",
     xaxis = dict(title = '날짜', showgrid = False),
     yaxis = dict(title = '확진자수'),
     margin = dict(t = 50, b = 25, l =  25, r =  25),
     showlegend = False)

{'title': '코로나19 발생 현황',
 'xaxis': {'title': '날짜', 'showgrid': False},
 'yaxis': {'title': '확진자수'},
 'margin': {'t': 50, 'b': 25, 'l': 25, 'r': 25},
 'showlegend': False}

# 2. Trace

Trace: 데이터를 시각화한 도형 레이어

## 데이터 로드 및 전처리

In [27]:
# 구글드라이브에서 데이터 불러오기
## 출처 : https://kess.kedi.re.kr/contents/dataset?itemCode=04&menuId=m_02_04_03_02&tabId=m3, 학과별, 2020년도
### NOTE : 해당 파일(주피터 노트북, ipynb 파일)을 구글코랩 환경이 아닌 다른 환경에서 실행하는 경우, os.chdir() 에 들어가는 작업디렉토리를 실습 데이터 (엑셀파일)가 위치한 디렉토리로 변경후 사용하시기 바랍니다.
### NOTE : 실습 데이터는 제공되며, 위 사이트에서 다운로드 받으실 수 있습니다.

#from google.colab import drive
#import os
#drive.mount('/content/drive')

#os.chdir("/content/drive/MyDrive/데이터가공과시각화[2]")
#file_path = "./2020년 학과별 고등교육기관 취업통계.xlsx"

#!pip3 install openpyxl

df_취업률 = pd.read_excel(
    "2020년 학과별 고등교육기관 취업통계.xlsx",
    sheet_name="학과별",
    skiprows=13,  # 첫 13행을 제외
    dtype={col: "string" for col in range(9)}  # 처음 9개 컬럼을 문자형으로 지정
)

df_취업률.iloc[:, 9:] = df_취업률.iloc[:, 9:].apply(pd.to_numeric, errors="coerce")

selected_columns = list(df_취업률.columns[:9]) + \
                   [col for col in df_취업률.columns if col.endswith("계")] + \
                   ["입대자"]

df_취업률 = df_취업률[selected_columns]

In [28]:
df_취업률_500 = df_취업률[df_취업률["졸업자_계"] < 500]
new_column_names = list(df_취업률_500.columns[:9]) + ['졸업자수', '취업률', '취업자수'] + list(df_취업률_500.columns[12:])
df_취업률_500.columns = new_column_names

In [29]:
df_취업률_500.head()

,조사기준일,학제,과정구분,대계열,중계열,소계열,학과코드,학과명,학위구분,졸업자수,...,1차 유지취업자_계,1차 유지취업률_계,2차 유지취업자_계,2차 유지취업률_계,3차 유지취업자_계,3차 유지취업률_계,4차 유지취업자_계,4차 유지취업률_계,입학당시 기취업자_계,입대자
0,2020.12.31,전문대학(4년제),전문대학과정,의약계열,간호,간호,C06010100004,간호학과,<NA>,391,...,298,95.2,292,93.3,280,89.5,277,88.5,63,1
1,2020.12.31,전문대학(4년제),전문대학과정,의약계열,간호,간호,C06010100008,간호학과(4년제),<NA>,482,...,377,97.2,362,93.3,347,89.4,342,88.1,38,0
2,2020.12.31,전문대학(3년제),전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100002,관광일본어과,<NA>,31,...,7,87.5,7,87.5,7,87.5,7,87.5,2,4
3,2020.12.31,전문대학(3년제),전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100003,관광일본어전공,<NA>,1,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
4,2020.12.31,전문대학(3년제),전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100005,관광일어과,<NA>,74,...,16,88.9,11,61.1,8,44.4,8,44.4,5,1


## Trace를 추가하는 방식

In [30]:
# 방법1
fig = go.Figure()

fig.add_trace(
    {         # 트레이스 추가
    'type' : 'scatter', # scatter 트레이스
    'mode' : 'markers', # markers 모드
    'x' : df_취업률_500['졸업자수'],
    'y' : df_취업률_500['취업자수'],
    'marker' : { 'size' : 3, 'color' : 'darkblue'} # marker 크기와 색상 설정
    }
)

In [31]:
# 방법 2
fig = go.Figure()

fig.add_trace(go.Scatter({
    'mode' : 'markers',
    'x' : df_취업률_500['졸업자수'],
    'y' : df_취업률_500['취업자수'],
    'marker' : {'size' : 3, 'color' : 'darkblue'}
    }))

In [32]:
# 방법 3
fig = go.Figure()

fig.add_trace(go.Scatter(
    mode = 'markers',
    x = df_취업률_500['졸업자수'],
    y = df_취업률_500['취업자수'],
    marker = dict(size = 3, color = 'darkblue')
))

fig.show()

## 기본적인 속성

In [33]:
# name
fig = go.Figure()

for 대계열, group in df_취업률_500.groupby('대계열'):
    fig.add_trace({
        'type' : 'scatter',
        'mode' : 'markers',
        'x': group['졸업자수'],
        'y': group['취업자수'],
        'name' : 대계열}
    )

fig.show()

In [34]:
# opacity
fig = go.Figure()

fig.add_trace({
        'type' : 'scatter',
        'mode' : 'markers',
        'x': df_취업률_500['졸업자수'],
        'y': df_취업률_500['취업자수'],
        'marker' : {'opacity' : 0.3}}
    )

fig.show()

In [35]:
# showlegend
fig = go.Figure()

for 대계열, group in df_취업률_500.groupby('대계열'):
    fig.add_trace({
        'type' : 'scatter',
        'mode' : 'markers',
        'x': group['졸업자수'],
        'y': group['취업자수'],
        'name' : 대계열,
        'showlegend': False}
    )

fig.show()

In [36]:
# update_layout
## 전체 범례 설정
fig.update_layout(
    title = {'text': '계열별 졸업자수 vs 취업자수',
             'x' : 0.5,
             'xanchor' : 'center',
             'yanchor' : 'top'},
    legend = dict(orientation = 'h', # Dictionary를 만드는 다른 방법
                  bordercolor = 'gray',
                  borderwidth = 2,
                  x = 0.1,
                  y = 0.95,
                  xanchor = 'left'),
    showlegend = True)

fig.show()

## 데이터값을 표시하기 위한 속성

In [37]:
# text
temp = df_covid19_100.groupby('location').agg(new_cases = ('new_cases', 'sum'))
temp

,new_cases
location,
한국,34132989.0
아시아,225441549.0
유럽,190739108.0
북미,72675836.0
남미,30829052.0
아프리카,4754188.0
오세아니아,14210403.0


In [38]:
fig = go.Figure()
fig.add_trace({
    'type' : 'bar', # bar plot
    'x': temp.index,
    'y': temp['new_cases'],
    'text' : temp['new_cases']
})

fig.show()

In [39]:
# textposition
fig = go.Figure()

fig.add_trace({ # bar plot
    'type' : 'bar',
    'x': temp.index,
    'y': temp['new_cases'],
    'text' : temp['new_cases'],
    'textposition' : 'outside'}) ## textposition: 'auto', 'none', 'inside'

fig.show()

In [71]:
# texttemplate
fig = go.Figure()

fig.add_trace({
    'type' : 'bar',
    'x': temp.index,
    'y': temp['new_cases'],
    'text' : temp['new_cases'],
    'textposition' : 'outside',
    'texttemplate' : '확진자수: %{text:,}' # 콤마가 포함된 숫자
}) # %{text:2f} 등

fig.show()

## 호버(hover) 관련 속성

In [72]:
# hoverinfo
fig = go.Figure()

fig.add_trace({
    'type' : 'scatter',
    'mode' : 'markers',
    'x': df_취업률_500['졸업자수'],
    'y': df_취업률_500['취업자수'],
    'hoverinfo' : 'y'}) ## hover 설정

fig.show()

In [73]:
# hovertext
fig = go.Figure()

fig.add_trace({
    'type' : 'scatter',
    'mode' : 'markers',
    'x': df_취업률_500['졸업자수'],
    'y': df_취업률_500['취업자수'],
    # hovertext의 설정
    'hovertext' : '중계열:'+ df_취업률_500['중계열']+'<br>'+'소계열:'+df_취업률_500['소계열']
})

fig.show()

In [74]:
# hovertemplate
fig = go.Figure()

fig.add_trace({
    'type' : 'scatter',
    'mode' : 'markers',
    'x': df_취업률_500['졸업자수'],
    'y': df_취업률_500['취업자수'],
    'hovertext' : df_취업률_500['중계열'],
    # hovertemplate의 설정
    'hovertemplate' : ' 졸업자:%{x}, 취업자:%{y}, 대계열:%{hovertext}'
})

fig.show()

# 3. Layout

## 기본속성

In [44]:
# title
fig_scatter = go.Figure()

fig_scatter.add_trace({
    'type' : 'scatter',
    'mode' : 'markers',
    'x' : df_취업률_500['졸업자수'],
    'y' : df_취업률_500['취업자수'],
    'marker' : {'color' : 'darkblue'}
})

fig_scatter.update_layout(
    title = dict(text = "<b>졸업자 대비 취업자수</b>",
                 x = 0.5,
                 y = 0.9,
                 xanchor = 'center',
                 yanchor = 'bottom')
)

fig_scatter.show()

In [45]:
# papar_bgcolor
fig_scatter.update_layout(paper_bgcolor = 'lightgray')

In [46]:
# plot_bgcolor
fig_scatter.update_layout(plot_bgcolor = 'lightgray')

## 축의 설정

In [47]:
# xaxis, yaxis
fig_scatter.update_layout(
    xaxis = dict(
        title = '<b>학과 졸업자수</b><sub>(명)</sub>',
        color = 'black',
        zerolinecolor = 'black',
        zerolinewidth = 3,
        gridcolor = 'gray',
        gridwidth = 1),
    yaxis = dict(
        title = '<b>학과 취업자수</b><sub>(명)</sub>',
        color = 'black',
        zerolinecolor = 'black',
        zerolinewidth = 3,
        gridcolor = 'gray',
        gridwidth = 1)
)

fig_scatter.show()

In [48]:
# 눈금의 조정
fig_scatter.update_layout(
    xaxis = dict(tickmode = 'array',                    # tickmode를 "array"로 설정
                ticktext = ('소규모', '중규모', '대규모'),   # ticktext 설정
                tickvals = (100, 300, 400)              # tickvals 설정
                 ),
    yaxis = dict(tickmode = 'linear', ## tickmode를 "linear"로 설정
                tick0 = 0,            ## tick0 설정
                dtick = 100)          ## dtick 설정
)

fig_scatter.show()

In [49]:
# 축의 범위 조정
fig_scatter = go.Figure(fig_scatter)

fig_scatter.update_layout(
    xaxis = dict(range = (0, 350),          # X축 range (튜플)
                rangemode = 'nonnegative'), # X축 rangemode
    yaxis = dict(range = [0, 300],          # Y축 range (리스트)
                rangemode = 'tozero'),      # Y축 rangemode
    margin = dict(pad = 5)
    )

fig_scatter.show()

In [50]:
# font
fig_scatter.update_layout(font = dict(family = "나눔고딕",
                                      color = 'Crimson',
                                      size = 20))

In [75]:
# legend
df_covid19_100_wide.head()

,date,확진자_한국,확진자_아시아,확진자_유럽,확진자_북미,확진자_남미,확진자_아프리카,확진자_오세아니아,백신접종완료자_한국,백신접종완료자_아시아,백신접종완료자_유럽,백신접종완료자_북미,백신접종완료자_남미,백신접종완료자_아프리카,백신접종완료자_오세아니아
0,2021-10-03,16224.0,1019817.0,909344.0,923259.0,169100.0,69112.0,20191.0,51.97,37.06,51.96,48.33,41.82,4.30,32.51
1,2021-10-10,13039.0,905692.0,1005831.0,819132.0,153169.0,53609.0,19859.0,58.56,38.36,52.73,49.34,44.15,4.56,36.25
2,2021-10-17,10629.0,843548.0,1159084.0,691289.0,124720.0,46062.0,18904.0,63.84,39.25,53.26,50.25,46.05,4.88,40.29
3,2021-10-24,9644.0,776191.0,1437425.0,597748.0,137108.0,39367.0,18688.0,69.23,41.02,53.69,51.14,47.90,5.23,43.57
4,2021-10-31,13296.0,745385.0,1590405.0,603822.0,136522.0,37830.0,13886.0,74.40,42.35,54.36,52.72,49.99,5.66,46.31


In [76]:
# 예시 : 코로나 확진자 추이
fig_line = go.Figure()

for code in ["한국", "아시아", "유럽", "북미"]:
    fig_line.add_trace({
        'type' : 'scatter',
        'mode' : 'lines',
        'x' : df_covid19_100_wide.index,
        'y' : df_covid19_100_wide['확진자_'+code],
        'name' : code
})

fig_line.show()

In [53]:
# 범례의 편집
# 범례 layout 설정
fig_line.update_layout(
    title = dict(text = '대륙별 신규 확진자수 추이',
                 x = 0.5,
                 xanchor = 'center',
                 yanchor = 'top'),
    legend = dict(orientation = 'h',
                  bordercolor = 'gray',
                  borderwidth = 2, x = 0.95,
                  y = 0.95,
                  xanchor = 'right'),
    showlegend = True)
fig_line.show()


# 4. Subplot

In [77]:
from plotly.subplots import make_subplots

In [78]:
subplot_titles = ('한국', '아시아', '유럽', '북미', '남미', '오세아니아', '아프리카')

fig_subplot = make_subplots(
    rows = 3, cols = 3,
    column_widths = [1/3, 1/3, 1/3], ## 열 방향 너비 설정
    row_heights =   [1/3, 1/3, 1/3], ## 행 방향 너비 설정
    subplot_titles = subplot_titles)


In [79]:
# Subplot1 - 한국
fig_subplot.add_trace({
    'type' : 'scatter',
    'mode' : 'lines',
    'x': df_covid19_100_wide.index,
    'y': df_covid19_100_wide['확진자_한국'],
    'name':'한국'},
    row = 1, col = 1)

# Subplot2 - 아시아
fig_subplot.add_trace({
    'type' : 'scatter',
    'mode' : 'lines',
    'x': df_covid19_100_wide.index,
    'y': df_covid19_100_wide['확진자_아시아'],
    'name':'아시아'},
    row = 1, col = 2)

# Subplot3 - 유럽
fig_subplot.add_trace({
    'type' : 'scatter',
    'mode' : 'lines',
    'x': df_covid19_100_wide.index,
    'y': df_covid19_100_wide['확진자_유럽'],
    'name':'유럽'},
    row = 1, col = 3)

# Subplot4 - 북미
fig_subplot.add_trace({
    'type' : 'scatter',
    'mode' : 'lines',
    'x': df_covid19_100_wide.index,
    'y': df_covid19_100_wide['확진자_북미'],
    'name':'북미'},
    row = 2, col = 1)

# Subplot5 - 남미
fig_subplot.add_trace({
    'type' : 'scatter',
    'mode' : 'lines',
    'x': df_covid19_100_wide.index,
    'y': df_covid19_100_wide['확진자_남미'],
    'name':'남미'},
    row = 2, col = 2)

# Subplot6 - 아프리카
fig_subplot.add_trace({
    'type' : 'scatter',
    'mode' : 'lines',
    'x': df_covid19_100_wide.index,
    'y': df_covid19_100_wide['확진자_아프리카'],
    'name':'아프리카'},
    row = 2, col = 3)

# Subplot7 - 오세아니아
fig_subplot.add_trace({
    'type' : 'scatter',
    'mode' : 'lines',
    'x': df_covid19_100_wide.index,
    'y': df_covid19_100_wide['확진자_오세아니아'],
    'name':'오세아니아'},
    row = 3, col = 1)

fig_subplot.update_layout(title = dict(text = "최근 100일간 코로나19 확진자", x = 0.5),
                          showlegend = False)

fig_subplot.show()

In [80]:
fig_subplot = make_subplots(
    rows=4,
    cols=3,
    specs = [
             [{'colspan' : 3, 'rowspan' : 2}, None, None],
             [None, None, None],
             [{}, {}, {}],
             [{}, {}, {}]
    ],
    shared_xaxes = True,
    shared_yaxes = True,
    subplot_titles = ('한국', '아시아', '유럽', '북미', '남미', '오세아니아', '아프리카')
)

In [81]:
locations = ['한국', '아시아', '유럽', '북미', '남미', '오세아니아', '아프리카']
row_col = [(1,1), (3,1), (3,2), (3,3), (4,1), (4,2), (4,3)]

for i, location in enumerate(locations):
  fig_subplot.add_trace(go.Scatter(
      x=df_covid19_100_wide.index,
      y=df_covid19_100_wide[f'확진자_{location}'],
      mode='lines',
      name=location
  ), row=row_col[i][0], col=row_col[i][1])

fig_subplot.update_layout(title = dict(text = "최근 100일간 코로나19 확진자", x = 0.5),
                          showlegend = True)

fig_subplot.show()

In [59]:
# colorscale : 색상의 조절
fig = go.Figure()

fig.add_trace({
    'type' : 'scatter',
    'mode' : 'markers',
    'x': df_취업률_500['졸업자수'],
    'y': df_취업률_500['취업자수'],
    'marker' : {'color' : df_취업률_500['취업률'], 'colorscale' : 'blues'}})

fig.show()

In [60]:
fig = go.Figure()

fig.add_trace({
    'type' : 'scatter',
    'mode' : 'markers',
    'x': df_취업률_500['졸업자수'],
    'y': df_취업률_500['취업자수'],
    'marker' : {'color' : df_취업률_500['취업률'],
                'colorscale' : 'Blues',
                'cmin' : 50,
                'cmax' : 100,
                'colorbar' : {'title' : '취업률', 'ticksuffix' : '%'},
                'reversescale' : True}})

fig.show()

# 5. 배포하기

Google Golab 에서 호환성 이슈로 `Ch.5 : 배포하기` 중 `kaleido` 파트 생략

In [62]:
!pip3 install chart_studio

# chart studio에 접속
import chart_studio
chart_studio.tools.set_credentials_file(username = 'sjshin',
                                        api_key = 'kFiv8hiw0t4PLVjxvow6')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [chart_studio]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [63]:
import chart_studio.plotly as py
py.plot(fig, filename = "job", auto_open = True) # 차트스튜디오에 업로드

AttributeError: 'str' object has no attribute 'stable_semver'